In [1]:
from crewai import Agent, Task, Crew
import os
import re

from langchain_openai import ChatOpenAI
# Load transcript


# Load local LLM
os.environ["OPENAI_API_KEY"]="89g8df0gkjgjfe"

llm = ChatOpenAI(
    model="ollama/phi3.5:3.8b",  # or your preferred model
    base_url="http://localhost:11434/v1"  # Make sure Ollama is running
)
transcript_path = "C:/Users/umall/Documents/github_projects/ScribeX - Agentic/data/formatted_transcript/Jane Doe_Dr. John Smith_2025-05-28.txt"
assert os.path.exists(transcript_path), "Transcript file missing"

with open(transcript_path, "r") as f:
    transcript = f.read()

subjective_agent = Agent(name="Subjective Agent", llm=llm,role="Symptom gatherer", goal="Extract subjective symptoms", backstory="You are a compassionate medical assistant who specializes in capturing patients' subjective experiences. You are trained to interpret how patients describe their symptoms, including onset, duration, and intensity. You listen attentively and do not make assumptions — you only record what the patient says, in their own words where appropriate.")
objective_agent = Agent(name="Objective Agent", llm=llm,role="Observer", goal="Extract objective findings", backstory="You are a detail-oriented nurse practitioner who records clinical observations and vital signs. You only note down objective facts from the transcript, such as physical findings or measurements the doctor mentions. You avoid interpreting — your role is to report what was seen, heard, or measured.")
assessment_agent = Agent(name="Assessment Agent",llm=llm, role="Medical summarizer", goal="Generate assessment", backstory="You are a clinical reasoning expert trained in medical summarization. Based on both subjective and objective inputs, your job is to synthesize the doctor’s findings and form a clinical impression. You do not generate new diagnoses unless explicitly mentioned — instead, you summarize the condition and its likely cause based on the transcript.")
plan_agent = Agent(name="Plan Agent",llm=llm, role="Planner", goal="Suggest next steps", backstory="You are a diligent junior physician who assists in treatment planning. You summarize what the doctor advised the patient, including medications, further tests, referrals, and follow-up plans. You do not invent new steps, only extract what was actually recommended in the transcript.")

subjective_task=Task(
            description=f"From this text:\n{transcript}\n\nWrite the SUBJECTIVE section in SOAP format.if not able to conclude mention that",
            agent=subjective_agent,
            expected_output="Subjective section capturing patient's symptoms and personal experience"
        )
objective_task=Task(
            description=f"From this text:\n{transcript}\n\nWrite the OBJECTIVE section in SOAP format.if not able to conclude mention that",
            agent=objective_agent,
            expected_output="Objective clinical observations and measurements from the doctor"
        )

assessment_task=Task(
            description=f"From this text:\n{transcript}\n\nWrite the ASSESSMENT section in SOAP format. if not able to conclude mention that",
            agent=assessment_agent,
            expected_output="Doctor’s assessment or diagnosis of the condition"
        )
plan_task=Task(
            description=f"From this text:\n{transcript}\n\nWrite the PLAN section in SOAP format.if not able to conclude mention that",
            agent=plan_agent,
            expected_output="Plan of care including prescriptions, tests, and follow-up instructions"
        )

# 2. Define tasks for each agent

# 3. Run Crew
def run_agentic_soap():
    #[subjective_task, objective_task, assessment_task, plan_task]
    crew = Crew(tasks=[subjective_task], agents=[subjective_agent])
    results = crew.kickoff()
    return results




In [2]:
soap_sections = run_agentic_soap()
# Save to file or display


In [ ]:
for x in soap_sections:
    print(x[0]=='raw'
    print(x[1])

raw
Subjective (SOAP):

S: The patient reports experiencing breathlessness while lying down for two months. They describe this symptom as a significant challenge that has impacted their ability to perform daily activities and work, particularly when exerting themselves at home or standing around all day after returning from employment where they serve as a cashier (Subjective Fatigue).

The patient also reports lower extremity swelling which is localized in one leg with associated pain. The location of the edema has not changed and includes areas such as calves, ankles, feet up to shins, worsening during evenings after work hours (Subjective Edema). They have noticed this over a period longer than two months but for specificity purposes in SOAP format we can consider it ongoing.

The patient confirms having had heart-related health issues; they suffered from a myocardial infarction approximately four years ago, leading to stenting and no subsequent surgeries since then (Subjective Past

In [ ]:
with open("C:/Users/umall/Documents/github_projects/ScribeX - Agentic/data/SOAP_NOTES/agentic.txt", "w", encoding="utf-8") as f:
    for section in soap_sections:
        print("section",section)
        if section[1] is not None:
            try:
                print
            except Exception as e:
                print(f"[ERROR] Could not write section: {e}")
                print(f"Section data: {section}")